# Lab 08 — Sequence architectures and forecasting protocol

**Dataset:** [tulipa762/electricity_load_diagrams](https://huggingface.co/datasets/tulipa762/electricity_load_diagrams). The main experiment is about **evaluation discipline**: chronological split versus random split, then one-step versus recursive multi-step forecasting.

## The math behind the experiment

For a one-step forecast, the model estimates

$$
\hat{y}_{t+1}=f_\theta(x_{t-k+1:t}).
$$

A common regression loss is

$$
MSE=\frac{1}{N}\sum_{i=1}^{N}(y_i-\hat{y}_i)^2.
$$

The critical experimental constraint is chronological ordering: training examples must not contain information from the future evaluation period. A random split can make a forecasting system look better while quietly leaking temporal information.

In [ ]:
!pip -q install datasets scikit-learn pandas numpy torch
from datasets import load_dataset
import numpy as np,pandas as pd
raw=load_dataset("tulipa762/electricity_load_diagrams","uci",split="train[:1]"); row=raw[0]
vals=[]
for v in row.values():
 if isinstance(v,(list,tuple)): vals.extend(v)
 elif isinstance(v,(int,float)): vals.append(v)
vals=np.asarray(vals,dtype=np.float32); vals=vals[np.isfinite(vals)]; vals=(vals-vals.mean())/(vals.std()+1e-6)
T=24; X=np.array([vals[i:i+T] for i in range(len(vals)-T-1)]); y=vals[T:T+len(X)]; print(X.shape,y.shape)

## Step 1 — Baseline

Use the last observed value as a naive forecast. A neural model should beat this baseline to justify its complexity.

In [ ]:
from sklearn.metrics import mean_squared_error
split=int(.8*len(X)); Xtr,ytr=X[:split],y[:split]; Xte,yte=X[split:],y[split:]
naive=np.repeat(Xte[:,-1,None],1,axis=1).ravel(); print("chronological naive MSE",mean_squared_error(yte,naive))
# Intentionally invalid random split
rng=np.random.default_rng(4); idx=rng.permutation(len(X)); cut=int(.8*len(X)); tr,va=idx[:cut],idx[cut:]
rand_naive=np.repeat(X[va][:,-1,None],1,axis=1).ravel(); print("random-split naive MSE",mean_squared_error(y[va],rand_naive))

## Step 2 — Leakage experiment

The important result is not that one split has a smaller number. The question is whether the split corresponds to the deployment question: **predict future values using only information that would have existed at prediction time.**

In [ ]:
# A simple ridge autoregressive model makes the leakage protocol visible without long training.
from sklearn.linear_model import Ridge
m=Ridge(alpha=1.0).fit(Xtr,ytr); chron=mean_squared_error(yte,m.predict(Xte)); rand=mean_squared_error(y[va],m.predict(X[va]));
pd.DataFrame([["chronological",chron],["random",rand]],columns=["evaluation_protocol","MSE"])

## Step 3 — Recursive horizon

For a one-step model, recursively feed its own prediction back into the input window. Record error at horizons 1, 2, 4 and 8. Error accumulation is the mechanism to explain.

In [ ]:
x=Xte[0].copy(); rows=[]
for h in [1,2,4,8]:
 cur=x.copy(); preds=[]
 for _ in range(h):
  p=float(m.predict(cur.reshape(1,-1))[0]); preds.append(p); cur=np.r_[cur[1:],p]
 true=yte[:h]; rows.append([h,mean_squared_error(true,preds)])
pd.DataFrame(rows,columns=["horizon","recursive_MSE"])

## Visualize the evaluation contract — time order and forecast horizon

The first graph compares a chronological test split with an intentionally random split. The second shows how recursive prediction error changes with forecast horizon. These plots make two separate issues visible: whether evaluation matches deployment and whether small one-step errors compound over repeated predictions.

In [ ]:
import matplotlib.pyplot as plt

protocol_results = pd.DataFrame(
    [["chronological", chron], ["random split", rand]],
    columns=["protocol", "MSE"]
)
horizon_results = pd.DataFrame(rows, columns=["horizon", "recursive_MSE"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(protocol_results["protocol"], protocol_results["MSE"])
axes[0].set_title("Evaluation protocol matters")
axes[0].set_ylabel("MSE (lower is better)")
axes[0].grid(axis="y", alpha=.25)

axes[1].plot(horizon_results["horizon"], horizon_results["recursive_MSE"], marker="o")
axes[1].set_title("Recursive forecast error")
axes[1].set_xlabel("Forecast horizon (steps)")
axes[1].set_ylabel("MSE")
axes[1].grid(alpha=.25)
fig.tight_layout()
plt.show()

## Conclusion / answer key

Expected: chronological evaluation is the defensible primary result; random splitting can produce misleading evidence when temporal dependence exists. Recursive forecasting should generally accumulate error with horizon.

### Research extension
Compare Elman/Jordan-style state choices at matched parameter count and evaluate both one-step and multi-step error.